# Prophet + Stockcast: Inventory Simulation Lab
**One store · Five products · Step-by-step · Visual explanations**

This lab connects a Prophet demand forecast to inventory decisions using Stockcast.

## Learning path
1. Build a five-product store dataset and fit Prophet.
2. Understand `InventoryStateDataFrame` (on-hand, pipeline, backorders).
3. Fit `OrderUpToPolicy` (turn a demand target into an order).
4. Run `SimulationEngine` (advance inventory through time).
5. Inspect `SimulationResult` and evaluate with `InventoryEvaluator`.
6. Compare service-level policies and search for a lower-cost policy.

**Important:** Prophet forecasts expected demand; inventory planning needs uncertainty in *cumulative demand over the protection horizon*. We estimate that uncertainty by bootstrapping Prophet residuals. This is a teaching approximation, not a production-calibrated probabilistic forecast.

The notebook uses an observed holdout period as the demand the simulation experiences. Prophet is trained only on the preceding history, so the experiment resembles a backtest.

## 0. Install and import

Requires Python 3.10+. Run in a fresh environment. In Databricks, install these as notebook/cluster libraries if `%pip` is not available in your environment.

In [ ]:
%pip install -q prophet stockcast numpy pandas matplotlib seaborn

In [ ]:
import warnings, inspect, importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
rng = np.random.default_rng(SEED)

for pkg in ["prophet", "stockcast", "numpy", "pandas"]:
    try:
        print(pkg, metadata.version(pkg))
    except metadata.PackageNotFoundError:
        print(pkg, "not installed")

## 1. Create a one-store, five-product POS dataset

We generate 600 days of daily sales for five products. The final 56 days are held out for simulation. The generated data has weekly seasonality, a mild trend, random noise and occasional promotion-like demand pulses.

In a real project, replace this cell with your store × SKU POS data. Stockcast's `unique_id` identifies each product at the stocking location level.

In [ ]:
STORE = "STORE_001"
PRODUCTS = ["TEA_250G", "COFFEE_500G", "CEREAL_750G", "JUICE_1L", "SNACK_6PK"]
N_HISTORY, N_TEST = 544, 56
all_dates = pd.date_range("2024-01-01", periods=N_HISTORY + N_TEST, freq="D")

params = {
    "TEA_250G":    (18, 0.0004, 0.12, 3.0),
    "COFFEE_500G": (12, 0.0002, 0.18, 2.5),
    "CEREAL_750G": (24, -0.0001, 0.10, 4.0),
    "JUICE_1L":    (15, 0.0003, 0.22, 3.0),
    "SNACK_6PK":   (30, 0.0005, 0.15, 5.0),
}
frames = []
for product, (base, trend_rate, weekly_amp, noise_sd) in params.items():
    t = np.arange(len(all_dates))
    weekly = 1 + weekly_amp * np.sin(2*np.pi*all_dates.dayofweek.to_numpy()/7)
    trend = 1 + trend_rate*t
    promo = np.where((t % 91 >= 70) & (t % 91 < 77), 1.30, 1.0)
    y = np.maximum(0, base*weekly*trend*promo + rng.normal(0, noise_sd, len(t)))
    frames.append(pd.DataFrame({
        "store_id": STORE, "unique_id": product,
        "date": all_dates, "y": np.round(y, 1)
    }))
sales = pd.concat(frames, ignore_index=True)
train = sales[sales.date < all_dates[N_HISTORY]].copy()
test = sales[sales.date >= all_dates[N_HISTORY]].copy()

print("Training rows:", len(train), "| Holdout rows:", len(test))
display(sales.head())

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True)
for ax, product in zip(axes, PRODUCTS):
    d = sales[sales.unique_id == product]
    ax.plot(d.date, d.y, lw=.8)
    ax.axvline(all_dates[N_HISTORY], color="red", ls="--", alpha=.7)
    ax.set_title(product, loc="left")
    ax.set_ylabel("Units")
axes[-1].set_xlabel("Date (red line = start of holdout)")
plt.tight_layout()
plt.show()

## 2. Fit one Prophet model per product

Prophet expects `ds` and `y`. We fit on the training window and forecast the 56-day holdout. We record holdout WMAPE as a forecast diagnostic.

For a real deployment, use rolling-origin backtesting, tune Prophet per SKU where justified, and compare against simpler baselines.

In [ ]:
models, forecasts, residual_pools, forecast_scores = {}, {}, {}, []

for product in PRODUCTS:
    hist = train[train.unique_id == product][["date", "y"]].rename(columns={"date": "ds"})
    model = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=True,
        daily_seasonality=False,
        interval_width=0.90,
    )
    model.fit(hist)
    future = model.make_future_dataframe(periods=N_TEST, freq="D")
    fc = model.predict(future).tail(N_TEST).reset_index(drop=True)

    fitted = model.predict(hist[["ds"]])["yhat"].to_numpy()
    residual_pools[product] = hist["y"].to_numpy() - fitted
    actual = test[test.unique_id == product].sort_values("date")["y"].to_numpy()
    pred = fc["yhat"].clip(lower=0).to_numpy()
    wmape = np.abs(actual-pred).sum() / max(actual.sum(), 1e-9)

    models[product], forecasts[product] = model, fc
    forecast_scores.append({"unique_id": product, "holdout_WMAPE": wmape})

display(pd.DataFrame(forecast_scores).sort_values("holdout_WMAPE"))

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 11), sharex=True)
for ax, product in zip(axes, PRODUCTS):
    actual = test[test.unique_id == product].sort_values("date")
    fc = forecasts[product]
    ax.plot(actual.date, actual.y, label="Observed holdout", lw=1)
    ax.plot(fc.ds, fc.yhat.clip(lower=0), label="Prophet yhat", lw=1.5)
    ax.fill_between(fc.ds, fc.yhat_lower.clip(lower=0), fc.yhat_upper.clip(lower=0),
                    alpha=.18, label="Prophet interval")
    ax.set_title(product, loc="left")
    ax.legend(loc="upper left", ncol=3)
plt.tight_layout()
plt.show()

## 3. Build uncertainty scenarios for the protection horizon

Assume:
- supplier lead time `L = 2` days;
- inventory review interval `R = 5` days;
- protection horizon `H = L + R = 7` days.

We sample historical Prophet residuals and add them to the next seven daily point forecasts. Each row of `paths` is one possible seven-day demand trajectory. We then take the 90th/95th/98th percentile of **cumulative seven-day demand** for each SKU.

Why cumulative? The order must protect the store across the entire time until another replenishment can arrive. A one-day P95 is not the right target for a seven-day protection window.

Bootstrap limitations: in-sample residuals can be optimistic and independent resampling can destroy serial correlation. For production, use rolling-origin errors and consider block bootstrap or a calibrated probabilistic model.

In [ ]:
LEAD_TIME = 2
REVIEW_PERIOD = 5
H = LEAD_TIME + REVIEW_PERIOD
N_PATHS = 5000
OPENING_DATE = pd.Timestamp(all_dates[N_HISTORY - 1])

demand_paths = {}
target_levels = {0.90: {}, 0.95: {}, 0.98: {}}

for product in PRODUCTS:
    mu = forecasts[product]["yhat"].head(H).clip(lower=0).to_numpy()
    residuals = residual_pools[product]
    sampled_errors = rng.choice(residuals, size=(N_PATHS, H), replace=True)
    paths = np.maximum(0, mu[None, :] + sampled_errors)
    demand_paths[product] = paths
    cumulative = paths.sum(axis=1)
    for level in target_levels:
        target_levels[level][product] = float(np.quantile(cumulative, level))

target_summary = pd.DataFrame([
    {"unique_id": p,
     "mean_H_day_demand": demand_paths[p].sum(axis=1).mean(),
     "P90": target_levels[.90][p],
     "P95": target_levels[.95][p],
     "P98": target_levels[.98][p]}
    for p in PRODUCTS
])
display(target_summary.round(1))

In [ ]:
p = PRODUCTS[0]
plt.figure(figsize=(10, 4))
plt.hist(demand_paths[p].sum(axis=1), bins=45, alpha=.8)
for q, color in [(0.90, "orange"), (0.95, "red"), (0.98, "purple")]:
    plt.axvline(target_levels[q][p], color=color, ls="--", label=f"P{int(q*100)} = {target_levels[q][p]:.1f}")
plt.title(f"{p}: simulated cumulative demand over {H}-day protection horizon")
plt.xlabel("Total demand (units)")
plt.ylabel("Scenario count")
plt.legend()
plt.show()

## 4. Concept 1 — `InventoryStateDataFrame`: what stock do we have?

This object is the inventory system's **state at one point in time**, one row per SKU. It tracks:
- `on_hand`: physical units available now;
- `in_transit`: an array of future receipts, indexed by arrival period;
- `backorders`: units owed to customers;
- `inventory_position = on_hand + sum(in_transit) - backorders`.

We start with eight days of recent average demand as opening stock. This is only a convenient teaching assumption; use actual counted inventory in a real project.

In [ ]:
from stockcast.core import InventoryStateDataFrame, SimulationEngine
from stockcast.policies import OrderUpToPolicy

opening_stock = []
for product in PRODUCTS:
    avg = train[train.unique_id == product].tail(28).y.mean()
    opening_stock.append(int(round(avg * 8)))

opening = pd.DataFrame({
    "unique_id": PRODUCTS,
    "date": OPENING_DATE,
    "on_hand": opening_stock,
})
inventory = InventoryStateDataFrame.from_observed(
    opening,
    allow_backorders=False,
)
display(inventory.get_dataframe())

### Inventory position — a worked mini-example

Suppose one SKU has 40 units on hand, 12 units arriving in two periods, and 3 units of backorders:

`inventory position = 40 + 12 - 3 = 49`

The store can sell only the 40 physical units today, but the replenishment policy sees the 49-unit position because the 12 are already ordered and 3 are owed.

In [ ]:
components = pd.Series({"On hand": 40, "In transit": 12, "Backorders": -3})
fig, ax = plt.subplots(figsize=(8, 3))
components.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", lw=.8)
ax.set_ylabel("Units")
ax.set_title("Inventory-position components (illustrative)")
plt.xticks(rotation=0)
plt.show()
print("Inventory position:", components.sum())

## 5. Concept 2 — `OrderUpToPolicy`: turn a target into an order

An order-up-to policy orders enough to bring inventory position up to target `S`:

`Q = max(0, S - inventory_position)`

The target `S` is the selected quantile of cumulative demand over the protection horizon. We use P95 for the main run.

The forecast-target table has one row per SKU. `date` marks the end of the protection window. `forecast_origin` tells Stockcast when the forecast was made.

In [ ]:
target_table = pd.DataFrame({
    "unique_id": PRODUCTS,
    "S": [target_levels[0.95][p] for p in PRODUCTS],
    "target_end": [OPENING_DATE + pd.Timedelta(days=H)] * len(PRODUCTS),
})

policy_95 = OrderUpToPolicy(
    lead_time=LEAD_TIME,
    review_period=REVIEW_PERIOD,
    freq="D",
    service_level=0.95,
    allow_backorders=False,
    date_column="target_end",
).fit(
    target_table,
    target_column="S",
    forecast_origin=OPENING_DATE,
)
display(target_table.round(1))

In [ ]:
decision = policy_95.predict(inventory, current_period=0)
display(decision)

**Inspect the decision:** Stockcast's policy computes an order decision; it does not directly make the supplier deliver the stock. The engine applies the order to the inventory pipeline. If the installed version returns an object with a tabular view, inspect it with `display(decision)` and `dir(decision)`.

A useful mental model:
- forecast/scenario layer supplies target demand coverage;
- policy layer compares target with inventory position;
- engine layer executes the order and later receives it.

## 6. Concept 3 — `SimulationEngine`: play the system forward

The engine is a time-step simulator. In each period it:
1. receives deliveries due now;
2. checks whether an order review is scheduled;
3. asks the policy for an order and places it in the pipeline;
4. serves demand (or records lost sales);
5. records the resulting inventory flows.

The run below uses the **observed holdout demand** as the demand path. The forecast was generated without seeing this holdout.

In [ ]:
future_demand = test[["unique_id", "date", "y"]].copy()
future_demand = future_demand.sort_values(["unique_id", "date"]).reset_index(drop=True)

engine = SimulationEngine(verbose=0)
result_95 = engine.run(
    policy=policy_95,
    demand_source=future_demand,
    inventory=inventory,
    random_seed=SEED,
)
print("Result type:", type(result_95))
print("Periods:", result_95.n_periods)
print("Policy:", result_95.policy_name)

## 7. Concept 4 — `SimulationResult`: inspect what happened

A result contains the run's history, event table, final inventory and run manifest. The event table is the detailed audit trail: what demand arrived, what was fulfilled, what was ordered and received, and what stock remained.

The exact event columns are versioned, so inspect them rather than assuming names.

In [ ]:
print("Result attributes:", [x for x in dir(result_95) if not x.startswith("_")])
print("Event frame columns:", result_95.to_event_frame().columns.tolist())
display(result_95.event_frame.head(12))

In [ ]:
event = result_95.to_event_frame().copy()
display(event.tail())
print("Run manifest:")
display(result_95.run_manifest)

### Visualize inventory and demand

The event table is the source of truth. The next cell prints its schema and attempts to plot the standard Stockcast fields. If your installed version uses different column names, adjust the mapping after inspecting `event.columns`.

In [ ]:
print(event.columns.tolist())

In [ ]:
# Stockcast's event table uses date/unique_id and records demand and stock flows.
# Adapt these names only if your installed release reports different names above.
fig, axes = plt.subplots(len(PRODUCTS), 1, figsize=(14, 11), sharex=True)
for ax, product in zip(axes, PRODUCTS):
    d = event[event["unique_id"] == product].sort_values("date")
    ax.plot(d["date"], d["ending_on_hand"], label="End-of-period on hand", lw=1.5)
    if "demand" in d.columns:
        ax.plot(d["date"], d["demand"], label="Demand", alpha=.5)
    ax.set_title(product, loc="left")
    ax.set_ylabel("Units")
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 8. `InventoryEvaluator`: measure business outcomes

Forecast metrics such as WMAPE answer: *how close was the forecast to actual demand?*

Inventory metrics answer: *what happened to customers and stock because of the forecast and policy?*

Examples:
- `fill_rate`: share of demand fulfilled;
- `avg_on_hand`: average physical inventory;
- `cycle_service_level`: share of periods without a shortage;
- `order_event_count`: number of order events.

A policy can have a higher fill rate but also hold more inventory. That is a business trade-off.

In [ ]:
from stockcast.evaluation import (
    InventoryEvaluator, fill_rate, avg_on_hand,
    cycle_service_level, order_event_count
)

metrics = [fill_rate, avg_on_hand, cycle_service_level, order_event_count]
evaluator = InventoryEvaluator().fit(result_95)
metrics_95 = evaluator.evaluate(
    metrics=metrics,
    context={"include_partial_cycles": False}
)
display(metrics_95.round(3))

## 9. Concept 5 — compare policies on identical demand

A fair experiment changes one thing at a time. We compare 90%, 95% and 98% protection targets while holding constant:
- the same observed holdout demand;
- the same opening inventory;
- the same lead time and review interval;
- the same scoring period.

`run_comparison` runs each fitted policy from its own copy of the opening state. This avoids one policy inheriting another policy's ending inventory.

In [ ]:
def make_policy(service_level):
    targets = pd.DataFrame({
        "unique_id": PRODUCTS,
        "S": [target_levels[service_level][p] for p in PRODUCTS],
        "target_end": [OPENING_DATE + pd.Timedelta(days=H)] * len(PRODUCTS),
    })
    return OrderUpToPolicy(
        lead_time=LEAD_TIME,
        review_period=REVIEW_PERIOD,
        freq="D",
        service_level=service_level,
        allow_backorders=False,
        date_column="target_end",
    ).fit(targets, target_column="S", forecast_origin=OPENING_DATE)

policies = {
    "P90": make_policy(0.90),
    "P95": make_policy(0.95),
    "P98": make_policy(0.98),
}
comparison = engine.run_comparison(
    policies=policies,
    demand_source=future_demand,
    inventory=inventory,
    random_seed=SEED,
)
display(comparison.summary())

In [ ]:
comparison_metrics = {}
for label in comparison:
    comparison_metrics[label] = InventoryEvaluator().fit(comparison[label]).evaluate(
        metrics=[fill_rate, avg_on_hand, cycle_service_level, order_event_count],
        context={"include_partial_cycles": False}
    ).iloc[0].to_dict()

comparison_df = pd.DataFrame(comparison_metrics).T
display(comparison_df.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison_df["fill_rate"].plot(kind="bar", ax=axes[0], title="Fill rate by policy")
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Share of demand fulfilled")
comparison_df["avg_on_hand"].plot(kind="bar", ax=axes[1], title="Average on-hand by policy")
axes[1].set_ylabel("Units")
plt.tight_layout()
plt.show()

## 10. Optimize: choose a policy using an explicit business objective

A higher service level is not automatically best. It usually requires more inventory.

To illustrate optimization, define a simple cost:
- lost-sales penalty: `$10` per unfulfilled unit;
- holding cost: `$0.05` per unit per day;
- fixed order cost: `$8` per order event.

These are placeholders. Replace them with your retailer/supplier economics. Stockcast can evaluate custom metrics and costs; here we show how to create a transparent cost function from event-level outcomes.

First inspect event columns. We use the standard event fields below; if your release names a flow differently, map it using the schema printed earlier.

In [ ]:
# Inspect the event schema before defining costs.
print(event.columns.tolist())

In [ ]:
# Cost illustration using evaluator outputs and event flows.
# Check event columns in your installed version before relying on this calculation.
cost_rows = []
LOST_SALE_COST = 10.0
HOLDING_COST = 0.05
ORDER_EVENT_COST = 8.0

for label in comparison:
    sim_result = comparison[label]
    ev = sim_result.event_frame
    ev_metrics = InventoryEvaluator().fit(sim_result).evaluate(
        metrics=[fill_rate, avg_on_hand, order_event_count]
    ).iloc[0]
    lost_units = float(ev["shortage_units"].sum())
    # Holding cost is charged on end-of-period units for each SKU-day.
    stock_unit_days = float(ev["ending_on_hand"].sum())
    order_events = float(ev["order_event_count"].sum())
    total_cost = (
        lost_units * LOST_SALE_COST
        + stock_unit_days * HOLDING_COST
        + order_events * ORDER_EVENT_COST
    )
    cost_rows.append({
        "policy": label, "fill_rate": ev_metrics["fill_rate"],
        "avg_on_hand": ev_metrics["avg_on_hand"], "order_events": order_events,
        "estimated_cost": total_cost
    })

cost_comparison = pd.DataFrame(cost_rows).set_index("policy")
display(cost_comparison.round(2))

The canonical event table uses `shortage_units`, `ending_on_hand`, and `order_event_count`. These are the source columns used in the cost calculation.

For a real optimization:
1. define holding, ordering, shortage, expiry and service-level economics;
2. run a grid or Bayesian search over policy parameters;
3. use multiple demand paths / historical backtest windows;
4. choose a robust policy, not merely the cheapest result on one lucky path;
5. validate on a final untouched period.

## 11. Scenario exercises

Try changing one condition at a time and rerun from the opening-state cell.

**A. Promotion shock**
- Multiply one product's holdout demand by 1.25.
- Does P95 still achieve the intended service level?

**B. Supplier delay**
- Change `LEAD_TIME` from 2 to 5.
- Recalculate `H = LEAD_TIME + REVIEW_PERIOD`.
- Regenerate cumulative protection targets. Do not reuse the old target.

**C. Service-level trade-off**
- Compare P90, P95 and P98.
- Plot fill rate against average on-hand.

**D. Case-pack constraints**
- Round order quantities up to multiples of 6 or 12 using Stockcast's ordering constraints, then measure the extra inventory and service impact.

**E. Better uncertainty**
- Replace the in-sample residual bootstrap with rolling-origin out-of-sample forecast errors or block-bootstrap residuals.

## 12. Summary: connect the concepts

| Concept | Plain-English meaning | Data Science analogy |
|---|---|---|
| `InventoryStateDataFrame` | Current stock position for every product | State snapshot |
| `OrderUpToPolicy` | Rule that calculates order quantity | Decision function |
| `SimulationEngine` | Advances the system through time | Time-stepped simulator |
| `SimulationResult` | Full record of the run | Experiment output / audit log |
| `InventoryEvaluator` | Measures service and inventory outcomes | Business KPI evaluator |
| `run_comparison` | Runs alternatives under same conditions | Controlled experiment |
| Optimization | Searches for good policy parameters | Hyperparameter / decision optimization |

### Official references
- [Stockcast Quickstart](https://filtheo.github.io/stockcast/get-started/quickstart/)
- [Inventory state](https://filtheo.github.io/stockcast/user-guide/inventory-state/)
- [Simulation engine](https://filtheo.github.io/stockcast/user-guide/engine/)
- [Evaluation](https://filtheo.github.io/stockcast/learn/07-evaluate/)
- [Compare forecasts and policies](https://filtheo.github.io/stockcast/how-to/compare-policies/)
- [Prophet documentation](https://facebook.github.io/prophet/)

**Practical caution:** this is an educational backtest. POS sales may be censored by stockouts, and residual bootstrapping may not preserve temporal dependence. Before operational use, validate demand uncertainty, lead-time distributions, ordering calendars, case packs, supplier constraints, costs and service-level definitions.